In [1]:
!pip install -q langchain==0.1.16 langchain-openai==0.0.8 langchain-community==0.0.32

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 817.7/817.7 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.1/303.1 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requir

In [2]:
!pip install --upgrade numpy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 55.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.0.32 requires numpy<2,>=1, but you have numpy 2.4.2 which is incompatible.
langchain 0.1.16 requires numpy<2,>=1, but you have numpy 2.4.2 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.2 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.2 which is incompatible.
xarray 2025.12.0 requires packaging>=24.1, but you have packaging 23.2 which is incompatible.
db-dtypes 1.5.0 requires packaging>=24.2.0, but you have packaging 23.2 which is incompatible.


In [3]:
from langchain_openai import ChatOpenAI
import os
import pandas as pd

from langchain.agents import AgentType, initialize_agent,load_tools
from langchain.tools import Tool


In [4]:
# # local model
# llm = ChatOpenAI(
#     openai_api_base="http://localhost:1234/v1",
#     openai_api_key="lm_studio",
#     model="llama-3.2-1b-instruct",
#     temperature=0.9
# )

In [5]:
llm = ChatOpenAI(
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key="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",
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0.5
)

os.environ["SERPAPI_API_KEY"] = "eb61dbeefd7de402fb004f38d59204ce3848e7052dda5ed0e4a5a06fbee5db97"


In [6]:
!git clone https://github.com/deepanrajm/deep_learning.git

Cloning into 'deep_learning'...
remote: Enumerating objects: 3150, done.
remote: Counting objects: 100% (125/125), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 3150 (delta 71), reused 69 (delta 30), pack-reused 3025 (from 2)
Receiving objects: 100% (3150/3150), 309.98 MiB | 26.38 MiB/s, done.
Resolving deltas: 100% (311/311), done.
Updating files: 100% (2653/2653), done.


In [7]:
EXCEL_PATH = "/content/deep_learning/agents/expense.xlsx"

In [18]:
def analyze_expenses_excel(query: str, currency_symbol: str = "$") -> str:
    """
    Analyze salary and expenses from Excel using pandas.
    """
    df = pd.read_excel(EXCEL_PATH)

    salary = df.loc[df["Category"].str.lower() == "salary", "Monthly_Amount"].sum()

    expenses_df = df[df["Category"].str.lower() != "salary"]

    total_expenses = expenses_df["Monthly_Amount"].sum()
    savings = salary - total_expenses
    savings_percentage = (savings / salary) * 100 if salary > 0 else 0

    top_expenses = (
        expenses_df
        .sort_values(by="Monthly_Amount", ascending=False)
        .head(3)[["Category", "Monthly_Amount"]]
        .to_dict(orient="records")
    )

    # Format the output string to include the currency symbol
    formatted_summary = (
        f"Monthly Salary: {currency_symbol}{salary:.2f}\n"
        f"Total Monthly Expenses: {currency_symbol}{total_expenses:.2f}\n"
        f"Monthly Savings: {currency_symbol}{savings:.2f}\n"
        f"Savings Percentage: {savings_percentage:.2f}%\n"
        "Top 3 Expense Categories:\n"
    )
    for item in top_expenses:
        formatted_summary += f"- {item['Category']}: {currency_symbol}{item['Monthly_Amount']:.2f}\n"

    return formatted_summary

In [19]:
from langchain_core.pydantic_v1 import BaseModel, Field

class ExpenseAnalyzerInput(BaseModel):
    query: str = Field(description="Should be a question or query about expenses.")
    currency_symbol: str = Field(description="The currency symbol to use for the output, e.g., '$' or '₹'. Defaults to '$'.", default="$")

expense_tool = Tool(
    name="expense_analyzer",
    description=(
        "Analyze salary and monthly expenses from Excel in the given path and return "
        "savings, spending breakdown, and high expense categories." +
        " Use this tool when the user asks for financial analysis based on an expense sheet." +
        " If the user specifies a currency, use the 'currency_symbol' parameter."
    ),
    func=analyze_expenses_excel,
    args_schema=ExpenseAnalyzerInput
)

In [10]:
pip install google-search-results

  Preparing metadata (setup.py) ... done
  Created wheel for google-search-results: filename=google_search_results-2.4.2-py3-none-any.whl size=32010 sha256=611cd00a4e2cc574a0cca077c255294dd99567c99d815a73fe3953037a0c95a3
  Stored in directory: /root/.cache/pip/wheels/0c/47/f5/89b7e770ab2996baf8c910e7353d6391e373075a0ac213519e
Successfully built google-search-results


In [11]:
tools = load_tools(
    ["serpapi", "llm-math"],
    llm=llm
)

tools.append(expense_tool)


In [12]:
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)


/usr/local/lib/python3.12/dist-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 0.2.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  warn_deprecated(


In [21]:
query = """
This is my monthly salary and expense sheet. Please analyze it and tell me:
1. How much I am saving
2. Where I am overspending
3. How I can reduce expenses
4. How to improve my savings
Please provide all monetary values in GBP.
"""

result = agent.run(query)
print(result)



> Entering new AgentExecutor chain...
Could not parse LLM output: {
  "action": "expense_analyzer",
  "action_input": "monthly_salary_expenses.xlsx"
}
Observation: Invalid or incomplete response
Thought:Question: This is my monthly salary and expense sheet. Please analyze it and tell me:
1. How much I am saving
2. Where I am overspending
3. How I can reduce expenses
4. How to improve my savings
Please provide all monetary values in GBP.

Thought: I need to use the `expense_analyzer` tool to read the Excel file `monthly_salary_expenses.xlsx` and obtain the required financial breakdown.

Action:
```
{
  "action": "expense_analyzer",
  "action_input": "monthly_salary_expenses.xlsx"
}
```
Observation: {'monthly_salary': np.int64(100000), 'total_monthly_expenses': np.int64(75000), 'monthly_savings': np.int64(25000), 'savings_percentage': np.float64(25.0), 'top_3_expense_categories': [{'Category': 'Rent', 'Monthly_Amount': 25000}, {'Category': 'Food', 'Monthly_Amount': 15000}, {'Category':